In [1]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str

In [4]:
def step_1(state: CrashState) -> CrashState:
    print("Step 1 executed")
    return {"step1": "done", "input": state["input"]}

In [7]:
def step_2(state: CrashState) -> CrashState:
   print("Step 2 is running... interrupt manually using the STOP button in the notebook toolbar if needed.")
   time.sleep(30) 
   return {"step2": "done"}

In [6]:
def step_3(state: CrashState) -> CrashState:
    print("Step 3 executed")
    return {"done": True}

In [8]:
graph = StateGraph(CrashState)
graph.add_node("step_1", step_1)
graph.add_node("step_2", step_2)
graph.add_node("step_3", step_3)

graph.set_entry_point("step_1")
graph.add_edge("step_1", "step_2")
graph.add_edge("step_2", "step_3")
graph.add_edge("step_3", END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [12]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    workflow.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
Step 1 executed
Step 2 is running... interrupt manually using the STOP button in the notebook toolbar if needed.
❌ Kernel manually interrupted (crash simulated).


In [14]:
print("Re-running the graph to demonstrate fault tolerance...")
final_state = workflow.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("Final State:", final_state)

Re-running the graph to demonstrate fault tolerance...
Step 2 is running... interrupt manually using the STOP button in the notebook toolbar if needed.
Step 3 executed
Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}
